

---

### File 1: `i2s_tx_demo.sv`

```systemverilog
// ============================================================================
// Module: i2s_tx_demo
//
// Description:
//   Demonstration audio transmitter implementing the Philips I2S standard.
//   Generates a digital sawtooth waveform (dual-mono) and serializes it
//   into a continuous 64-bit I2S frame (32 bits Left + 32 bits Right per sample).
//
// Protocol Characteristics (Philips I2S Standard):
//   1. Bit Clock (BCK):
//      - Serial data is driven by the transmitter on the FALLING edge of BCK.
//      - Serial data is sampled by the receiver on the RISING edge of BCK.
//   2. Word Select (WS / LRCK):
//      - WS = 0 indicates the Left audio channel.
//      - WS = 1 indicates the Right audio channel.
//   3. 1-Clock Delay (MSB Alignment):
//      - The WS line transitions one full bit-clock period BEFORE the MSB
//        of the new word is transmitted.
//      - This means the WS transition occurs simultaneously with the clock
//        cycle where the PREVIOUS word's LSB is sampled.
//
// Audio Format:
//   - 24-bit audio samples, MSB-aligned in 32-bit channel slots.
//   - Lower 8 bits of each 32-bit slot are zero-padded: {Sample[23:0], 8'b0}.
// ============================================================================

`timescale 1ns/1ps

module i2s_tx_demo #(
    // ------------------------------------------------------------------------
    // Parameters
    // ------------------------------------------------------------------------
    // Divider value to generate bit-clock (i2s_bck) from clk_30m:
    // Frequency(i2s_bck) = Frequency(clk_30m) / (2 * BCK_HALF_DIV).
    // Default: 30 MHz / (2 * 5) = 3 MHz BCK.
    parameter integer BCK_HALF_DIV = 5,

    // Initial 24-bit sample value for the sawtooth generator:
    parameter logic [23:0] SAW_START = 24'h800000, // Two's compliment most negative 24 bit number

    // Increment added to the sawtooth generator per stereo frame:
    parameter logic [23:0] SAW_STEP = 24'h010000 
) (
    // ------------------------------------------------------------------------
    // Ports
    // ------------------------------------------------------------------------
    input  logic clk_30m,      // Master input clock (nominal 30 MHz)
    input  logic reset_n,      // Active-low asynchronous reset
    output logic i2s_bck,      // I2S Bit Clock (BCK / SCK)
    output logic i2s_ws,       // I2S Word Select / Left-Right Clock (WS / LRCK)
    output logic i2s_rx_data   // I2S Serial Data line (SD / SDOUT)
);

    // Calculate minimum bit width required to hold (BCK_HALF_DIV - 1)
    localparam integer DIVIDER_WIDTH = (BCK_HALF_DIV < 2) ? 1 : $clog2(BCK_HALF_DIV);

    // ------------------------------------------------------------------------
    // Internal Signals and Registers
    // ------------------------------------------------------------------------
    // Counter to divide clk_30m down to generate i2s_bck half-periods
    logic [DIVIDER_WIDTH - 1:0] bck_divider;

    // Bit position counter within the 64-bit frame:
    //   0 to 31 : Left channel 32-bit slot
    //  32 to 63 : Right channel 32-bit slot
    logic [5:0] frame_bit;

    // Current 24-bit synthesized audio sample
    logic [23:0] sawtooth_value;

    // Full 32-bit formatted words containing 24-bit sample + 8-bit zero padding
    logic [31:0] left_word;
    logic [31:0] right_word;

    // Pre-calculated next sample value (wraps naturally on 24-bit overflow)
    logic [23:0] next_sawtooth;

    // Compute the next sawtooth sample combinationally
    assign next_sawtooth = sawtooth_value + SAW_STEP;

    // ------------------------------------------------------------------------
    // Sequential State Machine & Output Generation
    // ------------------------------------------------------------------------
    always_ff @(posedge clk_30m or negedge reset_n) begin
        if (!reset_n) begin
            // ----------------------------------------------------------------
            // Asynchronous Reset: Restore initial framing and audio states
            // ----------------------------------------------------------------
            bck_divider    <= '0;
            frame_bit      <= 6'd0;
            sawtooth_value <= SAW_START;

            // Align 24-bit sample to the top 24 bits of 32-bit word, zero-pad LSBs
            left_word      <= {SAW_START, 8'b0};
            right_word     <= {SAW_START, 8'b0};

            // Reset I2S bus lines:
            // BCK low, WS low (Left channel active), Data pre-loaded with MSB
            i2s_bck        <= 1'b0;
            i2s_ws         <= 1'b0;
            i2s_rx_data    <= SAW_START[23]; // MSB of the first left word

        end else if (bck_divider == BCK_HALF_DIV - 1) begin
            // Divider rollover: A half-period of i2s_bck has elapsed
            bck_divider <= '0;

            if (i2s_bck) begin
                // ============================================================
                // CURRENT STATE IS HIGH -> TRANSITIONING TO LOW (FALLING EDGE)
                // ============================================================
                // According to I2S specifications, the transmitter changes data
                // on the falling edge of BCK so that the data is stable and 
                // ready for the receiver to sample on the next rising edge.
                i2s_bck <= 1'b0;

                if (frame_bit == 6'd0) begin
                    // Transmit MSB of Left Channel (bit 31)
                    i2s_rx_data <= left_word[31];
                end else if (frame_bit <= 6'd31) begin
                    // Transmit remaining bits of Left Channel (bits 30 down to 0)
                    i2s_rx_data <= left_word[31 - frame_bit];
                end else if (frame_bit == 6'd32) begin
                    // Transmit MSB of Right Channel (bit 31)
                    i2s_rx_data <= right_word[31];
                end else begin
                    // Transmit remaining bits of Right Channel (bits 30 down to 0)
                    i2s_rx_data <= right_word[63 - frame_bit];

                    // When we drive the final bit (bit 0 / frame_bit 63) of the
                    // current frame, advance the audio generator to prepare the
                    // sample for the next upcoming frame.
                    if (frame_bit == 6'd63) begin
                        sawtooth_value <= next_sawtooth;
                        left_word      <= {next_sawtooth, 8'b0};
                        right_word     <= {next_sawtooth, 8'b0};
                    end
                end

            end else begin
                // ============================================================
                // CURRENT STATE IS LOW -> TRANSITIONING TO HIGH (RISING EDGE)
                // ============================================================
                // The receiver samples i2s_rx_data on this edge.
                // We advance the framing counter and toggle i2s_ws at boundaries.
                i2s_bck <= 1'b1;

                if (frame_bit == 6'd31) begin
                    // End of Left slot:
                    // In Philips I2S, WS transitions one clock period BEFORE
                    // the Right MSB. That transition occurs on this exact edge
                    // (coinciding with the sampling of Left word's LSB).
                    i2s_ws    <= 1'b1;     // Switch WS to Right channel (WS = 1)
                    frame_bit <= 6'd32;    // Advance to Right slot
                end else if (frame_bit == 6'd63) begin
                    // End of Right slot:
                    // WS transitions to Left channel (WS = 0) coinciding with
                    // the sampling of Right word's LSB.
                    i2s_ws    <= 1'b0;     // Switch WS to Left channel (WS = 0)
                    frame_bit <= 6'd0;     // Wrap counter to beginning of frame
                end else begin
                    // Normal bit advance within the current slot
                    frame_bit <= frame_bit + 1'b1;
                end
            end

        end else begin
            // Half-period counter increment
            bck_divider <= bck_divider + 1'b1;
        end
    end

endmodule
```

---

### File 2: `i2s_tb.sv`

```systemverilog
// ============================================================================
// Testbench: i2s_tb
//
// Description:
//   Self-checking verification environment for the `i2s_tx_demo` module.
//   Acts as an ideal I2S slave receiver:
//     - Samples data and word-select on the rising edge of i2s_bck.
//     - Checks that the word select (i2s_ws) matches Philips standard timing:
//         * Exactly 32 bits per channel slot.
//         * WS transitions at the slot boundary.
//         * WS = 0 for Left channel, WS = 1 for Right channel.
//     - Validates that the captured 32-bit serial words match the golden
//       reference data loaded from `expected_words.mem`.
// ============================================================================

`timescale 1ns/1ps

module i2s_tb;

    // ------------------------------------------------------------------------
    // Constants & Configuration
    // ------------------------------------------------------------------------
    localparam integer BCK_HALF_DIV = 2; // Accelerated BCK rate for faster sim
    localparam integer SLOT_COUNT    = 6; // Check 6 slots (3 stereo frames)

    // ------------------------------------------------------------------------
    // Signals
    // ------------------------------------------------------------------------
    // Primary clock and reset
    logic clk_30m = 1'b0;
    logic reset_n = 1'b0;

    // DUT I2S interconnects
    logic i2s_bck;
    logic i2s_ws;
    logic i2s_rx_data;

    // Golden reference memory array (read from external file)
    logic [31:0] expected_words [0:SLOT_COUNT - 1];

    // Shift register used to reconstruct 32-bit serial data
    logic [31:0] captured_word;

    // State tracking for edge detection and frame counting
    logic previous_ws = 1'b0;
    integer bit_count  = 0;
    integer word_count = 0;

    // ------------------------------------------------------------------------
    // Device Under Test (DUT) Instantiation
    // ------------------------------------------------------------------------
    i2s_tx_demo #(
        .BCK_HALF_DIV(BCK_HALF_DIV)
    ) dut (
        .clk_30m    (clk_30m),
        .reset_n    (reset_n),
        .i2s_bck    (i2s_bck),
        .i2s_ws     (i2s_ws),
        .i2s_rx_data(i2s_rx_data)
    );

    // ------------------------------------------------------------------------
    // Master Clock Generation (Period = 2 ns -> 500 MHz simulation clock)
    // ------------------------------------------------------------------------
    always #1 clk_30m = ~clk_30m;

    // ------------------------------------------------------------------------
    // Test Control, Initialization & Watchdog Timer
    // ------------------------------------------------------------------------
    initial begin
        // Load the golden test vectors generated by Python
        $readmemh("expected_words.mem", expected_words);

        // Hold reset active for 5 ns, then release
        #5 reset_n = 1'b1;

        // Watchdog timeout: Fail the simulation if it hangs or drops clocks
        #3000;
        $display("FAIL: timeout after %0d channel slots", word_count);
        $fatal(1);
    end

    // ------------------------------------------------------------------------
    // I2S Receiver & Protocol Checker
    // ------------------------------------------------------------------------
    // Evaluates every rising edge of BCK, matching real receiver hardware.
    always @(posedge i2s_bck) begin
        if (reset_n) begin
            // Shift incoming bit into the LSB of the captured word register
            captured_word = {captured_word[30:0], i2s_rx_data};

            // Detect transition on Word Select (end of previous slot)
            if (i2s_ws != previous_ws) begin
                // ------------------------------------------------------------
                // Check 1: Verify Channel Polarity
                // ------------------------------------------------------------
                // Philips format alternates:
                //   Slot 0 (Left finished)  -> WS changes to 1 (Right next)
                //   Slot 1 (Right finished) -> WS changes to 0 (Left next)
                // Formula: ((word_count % 2) == 0) evaluates to:
                //   word_count = 0 -> 1'b1 (expects WS = 1)
                //   word_count = 1 -> 1'b0 (expects WS = 0)
                if (i2s_ws !== ((word_count % 2) == 0)) begin
                    $display("FAIL: unexpected WS=%b at slot %0d", i2s_ws, word_count);
                    $fatal(1);
                end

                // ------------------------------------------------------------
                // Check 2: Verify Captured Word Against Golden Vector
                // ------------------------------------------------------------
                if (captured_word !== expected_words[word_count]) begin
                    $display("FAIL: slot %0d expected %h got %h",
                             word_count, expected_words[word_count], captured_word);
                    $fatal(1);
                end

                // Slot passed: Update counters and reset capture register
                word_count    = word_count + 1;
                bit_count     = 0;
                captured_word = '0;

                // Stop simulation once all configured slots have passed
                if (word_count == SLOT_COUNT) begin
                    $display("PASS: checked %0d I2S channel slots", SLOT_COUNT);
                    $finish;
                end

            end else begin
                // ------------------------------------------------------------
                // Check 3: Slot Length & Protocol Framing
                // ------------------------------------------------------------
                // A slot must contain exactly 32 clock cycles. If bit_count
                // reaches 31 without seeing a WS transition, the transmitter
                // violated the slot width timing.
                if (bit_count == 31) begin
                    $display("FAIL: slot %0d did not receive a Philips WS transition", word_count);
                    $fatal(1);
                end

                bit_count = bit_count + 1;
            end

            // Record current WS to detect edge transitions on next clock cycle
            previous_ws = i2s_ws;
        end
    end

endmodule
```